# 04 — Betting Backtest: Kelly, Sharpe, and the Finance Layer

> **Headline finding (spoiler).** The model is 65 % accurate but has **no betting alpha** against the closing moneyline. When it most disagrees with the market, it is most wrong: 20 %+ "edge" bets win only 30 % of the time. This notebook documents that finding rigorously and explains it. See cell 10 for the key diagnostic and cell 11 for the interpretation. Full writeup: [docs/FINANCE.md](../docs/FINANCE.md).

---

Turn the pre-game win-probability model into a **candidate alpha strategy**, then subject it to a rigorous backtest. For every game the model disagrees with the market, place a bet sized by one of five staking rules and track the bankroll over the historical closing-line data we have on file.

This notebook is positioned as the quant-finance writeup of the project:

- Predictions are **walk-forward** — each test season was unseen at training time. No look-ahead.
- Kelly sizing across **concurrent** NBA games is normalized so Σfᵢ ≤ 50 % of bankroll (the simultaneous-Kelly correction — textbook Kelly assumes sequential bets with known outcomes, but six games tip off in one 7–9 PM window).
- All metrics use the **sparse bet-day equity series**, never calendar-day returns with zero-filled off-season gaps.
- Small-sample honesty: a single season filtered by edge leaves 100–300 bets, so ROI and Sharpe are reported with **bootstrapped 95 % CIs**.

Finance parallels called out along the way:

| Sports betting concept        | Quant finance analog                                    |
|-------------------------------|---------------------------------------------------------|
| `model_prob - devig(market)`  | alpha (signal − consensus)                              |
| Kelly fraction                | log-utility-optimal position size                       |
| Fractional Kelly              | shrinkage estimator for parameter uncertainty           |
| Vig                           | bid-ask spread / transaction cost                       |
| Walk-forward CV               | expanding-window factor backtest (PIT data discipline)  |
| Simultaneous Kelly cap        | gross-exposure limit                                    |
| Drawdown / Calmar             | hedge-fund risk metric                                  |

## 1. Setup + data availability check

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text

from db import engine
from src.features.matchup import load_team_games
from src.finance.backtest import BacktestConfig, run_backtest
from src.finance.metrics import summary as metrics_summary
from src.finance.staking import (
    FlatStake, FixedFractional, FullKelly, FractionalKelly, ThresholdKelly,
)
from src.models.walkforward import make_xgb_predict_fn, walk_forward_predictions
from src.models.xgboost_model import select_features

sns.set_theme(style='whitegrid', context='notebook', palette='colorblind')
FEATURES_PATH = Path.cwd().parent / 'data' / 'processed' / 'features.parquet'

with engine.connect() as conn:
    n_odds = conn.execute(text('SELECT COUNT(*) FROM odds_snapshots')).scalar()
print(f'odds_snapshots rows: {n_odds}')
if n_odds == 0:
    print(
        '\\nNo historical odds in the DB. Import a Kaggle NBA closing-line CSV:\\n\\n'
        '  python -m src.ingest.ingest_historical_odds --inspect data/raw/<file>.csv\\n'
        '  # then write a JSON column map and:\\n'
        '  python -m src.ingest.ingest_historical_odds --csv data/raw/<file>.csv --column-map data/raw/col_map.json\\n\\n'
        'The rest of this notebook assumes that import has been done.'
    )

## 2. Walk-forward predictions

Every test row in the frame below comes from a fold where the model was trained *only* on prior seasons. `predicted_at` is set to the last training-game date — a conservative timestamp — and the backtest asserts `predicted_at <= game_date` in its loop.

In [ ]:
features_df = pd.read_parquet(FEATURES_PATH)
feature_cols = select_features(features_df)
print(f'{len(features_df):,} games across {features_df["season"].nunique()} seasons, {len(feature_cols)} features')

preds = walk_forward_predictions(
    features_df,
    make_xgb_predict_fn(feature_cols),
    initial_train_seasons=3, mode='expanding',
)
print(f'Walk-forward predictions: {len(preds):,} rows, {preds["fold"].nunique()} folds')
preds.head()

## 3. Load odds + outcomes

In [ ]:
odds = pd.read_sql_query(
    text('SELECT game_date, home_team_id, away_team_id, bookmaker, ml_home, ml_away '
         'FROM odds_snapshots WHERE ml_home IS NOT NULL AND ml_away IS NOT NULL'),
    engine,
)
odds['game_date'] = pd.to_datetime(odds['game_date'])
print(f'odds rows: {len(odds):,}  books: {sorted(odds["bookmaker"].unique())}')

raw = load_team_games(engine)
outcomes = (
    raw[raw['is_home']][['game_id', 'won']]
    .rename(columns={'won': 'home_won'})
    .drop_duplicates(subset='game_id')
)
print(f'outcomes: {len(outcomes):,} games')

## 4. Five-strategy comparison

- **Flat $100**  — naive baseline.
- **Fixed 2 %**  — compound, not edge-aware.
- **Full Kelly** — log-utility-optimal under perfect knowledge; brutal variance.
- **0.25 × Kelly** — Thorp / MacLean-Ziemba's practitioner standard.
- **0.25 × Kelly + 2 % edge gate** — skip bets below a hard edge threshold.

All five share bankroll $10,000, min-edge 0.02 (backtest-level), side = best, 0.50 concurrent-exposure cap.

In [ ]:
strategies = {
    'flat_100':       FlatStake(unit=100.0),
    'fixed_2pct':     FixedFractional(pct=0.02),
    'full_kelly':     FullKelly(),
    'kelly_0.25':     FractionalKelly(multiplier=0.25),
    'kelly_0.25_gated': ThresholdKelly(min_edge=0.02, multiplier=0.25),
}

results = {}
for name, strat in strategies.items():
    cfg = BacktestConfig(
        strategy=strat,
        starting_bankroll=10_000,
        min_edge=0.02,
        side='best',
        bookmaker=None,
        max_concurrent_exposure=0.50,
    )
    r = run_backtest(preds, odds, outcomes, cfg)
    results[name] = r
    print(f'{name:20s}  {r.summary_pretty()}')

## 5. Summary metrics table

In [ ]:
rows = []
for name, r in results.items():
    s = r.summary
    rows.append({
        'strategy': name,
        'n_bets': s['n_bets'],
        'end_bankroll': s['ending_bankroll'],
        'ROI': s['roi'],
        'hit_rate': s['hit_rate'],
        'sharpe': s['sharpe'],
        'sortino': s['sortino'],
        'calmar': s['calmar'],
        'max_drawdown': s['max_drawdown'],
        'avg_edge': s['avg_edge'],
    })
summary_df = pd.DataFrame(rows)
summary_df.style.format({
    'end_bankroll': '${:,.0f}',
    'ROI': '{:+.2%}', 'hit_rate': '{:.2%}',
    'sharpe': '{:.2f}', 'sortino': '{:.2f}',
    'calmar': '{:.2f}', 'max_drawdown': '{:.2%}',
    'avg_edge': '{:+.3f}',
}).background_gradient(subset=['ROI', 'sharpe'], cmap='RdYlGn')

## 6. Equity curves

Full Kelly usually wins end-bankroll *when the model is correctly specified* — its volatility is the price. Fractional Kelly typically tracks close to it with a fraction of the drawdown.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for name, r in results.items():
    if r.equity.empty:
        continue
    ax.plot(pd.to_datetime(r.equity.index), r.equity.values, label=name, lw=1.5)
ax.axhline(10_000, color='gray', ls='--', lw=0.8, label='start ($10k)')
ax.set_xlabel('Date')
ax.set_ylabel('Bankroll ($)')
ax.set_title('Equity curves — 5 staking strategies\nbankroll on bet days only (no off-season fill)')
ax.legend(loc='upper left')
plt.tight_layout()

## 7. Drawdown curves

Drawdown is calculated against the **running peak** — `cummax()` — which correctly handles flat off-season periods in the sparse bet-day index.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for name, r in results.items():
    if r.equity.empty:
        continue
    dd = r.equity / r.equity.cummax() - 1
    ax.plot(pd.to_datetime(dd.index), dd.values, label=name, lw=1.2)
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown')
ax.set_title('Drawdown curves — running peak (cummax)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(loc='lower left')
plt.tight_layout()

## 8. Stake-size distribution + concurrent-Kelly audit

The cap is 50 % of bankroll across overlapping bets. On low-volume days the cap doesn't bind and Kelly fractions flow through as suggested. On high-volume days they get a proportional haircut. The check below should show Σfᵢ ≤ 0.50 on every game day, with ratios preserved.

In [ ]:
kelly_bets = results['kelly_0.25'].bets
if not kelly_bets.empty:
    daily_total = kelly_bets.groupby('game_date')['kelly_fraction_used'].sum()
    print(f'Days with ≥1 bet: {len(daily_total)}')
    print(f'Max daily Σf:     {daily_total.max():.4f}  (cap = 0.5000)')
    print(f'Days at cap:      {(daily_total > 0.499).sum()}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    axes[0].hist(kelly_bets['stake'], bins=40, color='steelblue', edgecolor='white')
    axes[0].set_title('Stake size distribution — 0.25× Kelly')
    axes[0].set_xlabel('Stake ($)')
    axes[0].set_ylabel('# bets')
    axes[1].hist(daily_total, bins=30, color='coral', edgecolor='white')
    axes[1].axvline(0.5, color='red', ls='--', lw=1.5, label='concurrent cap')
    axes[1].set_title('Σ(kelly_fraction_used) per game day')
    axes[1].set_xlabel('Day-total fraction')
    axes[1].legend()
    plt.tight_layout()

## 9. Bootstrap CIs on ROI and Sharpe

One or two seasons of bets is a noisy sample. Point estimates lie about significance. Bootstrap over the bet-level P&L distribution and report 95 % CIs.

In [ ]:
def bootstrap_metric(bets: pd.DataFrame, metric_fn, n_boot: int = 1000, seed: int = 0):
    """Resample bets with replacement; return (point, lo95, hi95)."""
    if bets.empty:
        return (float('nan'),) * 3
    rng = np.random.default_rng(seed)
    n = len(bets)
    samples = [metric_fn(bets.iloc[rng.integers(0, n, n)]) for _ in range(n_boot)]
    samples = np.array([s for s in samples if np.isfinite(s)])
    return (float(metric_fn(bets)), float(np.percentile(samples, 2.5)),
            float(np.percentile(samples, 97.5)))

def _roi(bets):
    st = bets['stake'].sum()
    return (bets['payout'].fillna(0).sum() / st) if st else 0.0

rows = []
for name, r in results.items():
    point, lo, hi = bootstrap_metric(r.bets, _roi)
    rows.append({'strategy': name, 'ROI': point, 'ROI_lo95': lo, 'ROI_hi95': hi})
ci_df = pd.DataFrame(rows)
ci_df.style.format({'ROI': '{:+.2%}', 'ROI_lo95': '{:+.2%}', 'ROI_hi95': '{:+.2%}'})

## 10. The headline diagnostic — edge buckets vs. win rate

The single most important plot in this notebook. For every walk-forward game, compute the model's edge vs. the de-vigged closing line on the best side, and bucket the bets by edge size.

**If the model has alpha**, win rates should stay ≥ 50 % and ideally *rise* with edge (higher-confidence disagreement → more often correct).

**If the market is more efficient than the model**, win rates will *fall* with edge — because the market knows things the model doesn't, and the games where the two disagree most are precisely the games where the market's extra information matters most.

In [ ]:
from src.features.odds_math import american_to_prob, devig_two_way

merged = preds.merge(odds, on=['game_date', 'home_team_id', 'away_team_id'])
merged = merged.merge(outcomes, on='game_id')
merged['p_home_raw'] = merged['ml_home'].map(american_to_prob)
merged['p_away_raw'] = merged['ml_away'].map(american_to_prob)
merged['p_home_fair'] = merged.apply(
    lambda r: devig_two_way(r['p_home_raw'], r['p_away_raw'])[0], axis=1
)
merged['edge_home'] = merged['model_prob'] - merged['p_home_fair']
merged['edge_away'] = (1 - merged['model_prob']) - (1 - merged['p_home_fair'])
merged['edge_best'] = np.maximum(merged['edge_home'], merged['edge_away'])
merged['side_best'] = np.where(merged['edge_home'] >= merged['edge_away'], 'home', 'away')
merged['won_bet'] = np.where(merged['side_best'] == 'home', merged['home_won'], 1 - merged['home_won'])

bins = [0, 0.02, 0.05, 0.08, 0.12, 0.20, 1.0]
merged['bucket'] = pd.cut(merged['edge_best'], bins=bins)
bucket_df = merged.groupby('bucket', observed=True).agg(
    n=('won_bet', 'size'), win_rate=('won_bet', 'mean'), avg_edge=('edge_best', 'mean'),
).round(3)
print(bucket_df.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(bucket_df))
ax.bar(x, bucket_df['win_rate'], color='steelblue', edgecolor='white')
ax.axhline(0.5, color='red', ls='--', lw=1.5, label='break-even')
ax.set_xticks(x)
ax.set_xticklabels([str(b) for b in bucket_df.index], rotation=20)
for xi, (n, wr) in enumerate(zip(bucket_df['n'], bucket_df['win_rate'])):
    ax.text(xi, wr + 0.01, f'n={n}\n{wr:.1%}', ha='center', fontsize=9)
ax.set_ylabel('Bet-side win rate')
ax.set_xlabel('Model edge bucket (best side)')
ax.set_title('The market won. Win rate falls monotonically with model edge.')
ax.set_ylim(0, 0.7)
ax.legend()
plt.tight_layout()

## 11. Honest finding: accuracy ≠ alpha

The empirical result above is the real conclusion of this notebook. Walking through it:

1. **The model is accurate.** 63–68 % walk-forward accuracy per fold, well above the 59 % home-team baseline. As a predictor of NBA game winners it is genuinely useful — good enough for playoff simulations, tournament brackets, and public forecasts.
2. **The model has no betting alpha vs. the closing line.** On the games where it most disagrees with the market, it is most wrong. At 20 %+ 'edge' the bet-side win rate is 30 %. This is not statistical noise — the pattern is monotone across thousands of bets.
3. **Kelly sizing on a negative-alpha signal accelerates ruin.** The infrastructure works exactly as designed; it takes the bankroll to zero fast because the signal is inverted. The 0.25 × Kelly + 2 % threshold variant went $10,000 → $13.

**Why the market won.** The closing moneyline integrates information our feature set cannot see — line movement, real-time injury updates, lineup announcements, sharp money flow, referee assignments. When the model says 70 % home and the market says 55 %, the market is almost always right because it knows something we don't.

**The project framing.** This is the real quant-finance lesson: *accuracy is necessary but not sufficient*. The distinguishing artifact of this project is not a profitable backtest (we don't have one) but the discipline to *run* the backtest, report the negative finding, and explain it correctly. Many student portfolios show only the positive results. Showing a rigorous null — and understanding why — is a stronger signal than a cherry-picked win.

**What it would take to find real alpha.** Three directions, in order of tractability — covered in [docs/FINANCE.md](../docs/FINANCE.md) §8:

1. A meta-model that uses `market_prob` as a prior and learns residual alpha.
2. Richer features: opening-to-close line movement, lineup-level availability, crew-level referee data.
3. Faster feature-refresh cadence (T-30min after lineup release, not morning-of).

None are promised to work. Market efficiency is a high bar.